# So sánh ba nhánh: smoke và dev
T4, Internet bật; Input là ZIP đúng checksum. Không chạy training hoặc main.


In [ ]:
from pathlib import Path
import hashlib, json, os, stat, zipfile, subprocess, sys, time
from datetime import datetime, timezone
os.environ.update(CUDA_VISIBLE_DEVICES='0', USE_TF='0', USE_FLAX='0', PYTHONDONTWRITEBYTECODE='1', TOKENIZERS_PARALLELISM='false')
EXPECTED = '20c11d7069127e26492d4e46f4bc488fd8bacdafab889bbb21793d39ca6f5c54'
EXPECTED_TRANSPORT = '499e30ce3ff0490abf9e6c7c2c157d4457361588df4352c9ec5458b15da79ebc'
matches = [p for p in Path('/kaggle/input').rglob('runner-dev.zip') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED]
PACKAGE = Path('/kaggle/working/comparison-package-' + EXPECTED[:12])
if matches:
    assert len(matches) == 1, 'Duplicate matching ZIP inputs.'
    with zipfile.ZipFile(matches[0]) as z:
        names = z.namelist()
        assert len(names) == len(set(names))
        transport = z.read('bundle-manifest.json')
        assert hashlib.sha256(transport).hexdigest() == EXPECTED_TRANSPORT
        manifest = json.loads(transport)
        assert set(names) == set(manifest) | {'bundle-manifest.json'}
        assert all(not stat.S_ISLNK(i.external_attr >> 16) for i in z.infolist())
        payload = {name: z.read(name) for name in manifest}
else:
    manifests = [p for p in Path('/kaggle/input').rglob('bundle-manifest.json') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_TRANSPORT]
    assert len(manifests) == 1, 'Add the exact runner-dev ZIP or its extracted Kaggle Dataset.'
    source = manifests[0].parent.resolve()
    manifest = json.loads(manifests[0].read_text())
    payload = {}
    for name in manifest:
        p = (source / name).resolve()
        assert p.is_relative_to(source) and not (source / name).is_symlink()
        payload[name] = p.read_bytes()
for name, raw in payload.items():
    target = (PACKAGE / name).resolve()
    assert target.is_relative_to(PACKAGE.resolve()) and not Path(name).is_absolute()
    assert chr(92) not in name and ':' not in name
    assert len(raw) == manifest[name]['bytes'] and hashlib.sha256(raw).hexdigest() == manifest[name]['sha256']
for name, raw in payload.items():
    target = PACKAGE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == manifest[name]['sha256']
    else:
        with target.open('xb') as f: f.write(raw)
PYTHON = Path('/kaggle/working/comparison-env/bin/python')
print('Package authenticated. Main locked. Qwen weights not loaded.')


In [ ]:
subprocess.run([sys.executable, '-B', '-m', 'scripts.comparison.kaggle', '--package', str(PACKAGE)], cwd=PACKAGE, check=True)
print('Environment/assets checked. Smoke/dev Qwen has not run yet.')


In [ ]:
def execute(mode):
    run = Path('/kaggle/working/comparison-runs') / (mode + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    run.parent.mkdir(parents=True, exist_ok=True)
    console = Path(str(run) + '-console.log')
    started = time.monotonic()
    with console.open('x', encoding='utf-8') as log:
        process = subprocess.Popen([str(PYTHON), '-B', '-m', 'scripts.comparison.runner', '--package', str(PACKAGE), '--output', str(run), '--mode', mode], cwd=PACKAGE, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                progress = json.loads((run / 'progress.json').read_text())
            except (OSError, json.JSONDecodeError):
                progress = dict(phase='STARTING', rows=0)
            print(str(int(time.monotonic()-started)) + 's | ' + json.dumps(progress), flush=True)
            time.sleep(15)
    audit = json.loads((run / 'audit.json').read_text()) if (run / 'audit.json').exists() else None
    archive = Path(str(run) + '.zip')
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(console, 'console.log')
        for p in sorted(run.rglob('*')):
            if p.is_file(): z.write(p, p.relative_to(run).as_posix())
    receipt = dict(archive=archive.name, sha256=hashlib.sha256(archive.read_bytes()).hexdigest(), size_bytes=archive.stat().st_size, process_exit_code=process.returncode, audit=audit)
    receipt_path = Path(str(run) + '-receipt.json')
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
    print(json.dumps(receipt, indent=2))
    print('ZIP:', archive, '\nReceipt:', receipt_path, '\nAudit:', run / 'audit.json')
    assert process.returncode == 0 and audit is not None, 'Run incomplete; the ZIP and receipt preserve available failure evidence.'
    return run, audit
SMOKE_RUN, SMOKE_AUDIT = execute('smoke')


In [ ]:
assert SMOKE_AUDIT['rows'] == 3 and SMOKE_AUDIT['failure_rows'] == 0, 'Resolve smoke failures before dev.'
DEV_RUN, DEV_AUDIT = execute('dev')
